In [16]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.7 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1


In [17]:
!pip install requests pandas numpy tqdm torch torchvision scikit-learn

In [1]:
# Steam Games 2025 - Genre Classification from Cover Art
# Custom ResNet18 Training Notebook

# =========================
# 0. Imports
# =========================
import os
import ast
import requests
import pandas as pd
import numpy as np
from PIL import Image
from io import BytesIO
from tqdm import tqdm
tqdm.pandas()
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [2]:
# =========================
# 1. Load CSV
# =========================
DATA_PATH = "games_march2025_cleaned.csv"
df = pd.read_csv(DATA_PATH)

# Keep only relevant columns
df = df[["appid", "name", "header_image", "genres"]].dropna()

# Parse genre strings
# Some rows may be malformed, so we guard it

def safe_parse(x):
    try:
        return ast.literal_eval(x)
    except:
        return []


df["genres"] = df["genres"].apply(safe_parse)

# Remove empty genre rows
df = df[df["genres"].str.len() > 0]

In [3]:
# =========================
# 2. Build label space
# =========================
all_genres = sorted(set(g for gs in df["genres"] for g in gs))
genre_to_idx = {g: i for i, g in enumerate(all_genres)}
num_classes = len(all_genres)

print("Num genres:", num_classes)

# Multi-hot encoding

def encode(genres):
    vec = torch.zeros(num_classes)
    for g in genres:
        vec[genre_to_idx[g]] = 1
    return vec


df["label_vec"] = df["genres"].apply(encode)

Num genres: 33


In [4]:
# =========================
# 3. Download Images
# =========================
IMG_DIR = "steam_images"
os.makedirs(IMG_DIR, exist_ok=True)

def download_image(row):
    url = row["header_image"]
    path = os.path.join(IMG_DIR, f"{row['appid']}.jpg")

    if os.path.exists(path):
        return path

    try:
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            with open(path, "wb") as f:
                f.write(r.content)
            return path
    except:
        return None

    return None

print("Downloading images...")
df["image_path"] = df.progress_apply(download_image, axis=1)

# Drop failed downloads
df = df.dropna(subset=["image_path"]).reset_index(drop=True)

print("Final dataset size:", len(df))

100%|██████████| 89406/89406 [00:01<00:00, 45671.14it/s]

Final dataset size: 89403


In [5]:
# =========================
# 4. Train/Test Split
# =========================
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# =========================
# 5. Dataset
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class SteamDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = transform(image)

        label = row["label_vec"]

        return image, label

train_ds = SteamDataset(train_df)
test_ds = SteamDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=8, num_workers=2, pin_memory=True)

In [6]:
# =========================
# 6. Custom ResNet18 (your model)
# =========================
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.downsample is not None:
            identity = self.downsample(identity)

        out += identity
        return self.relu(out)

class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = nn.Sequential(
            BasicBlock(64, 64),
            BasicBlock(64, 64)
        )

        self.layer2 = nn.Sequential(
            BasicBlock(64, 128, 2, nn.Sequential(
                nn.Conv2d(64, 128, 1, 2, bias=False),
                nn.BatchNorm2d(128)
            )),
            BasicBlock(128, 128)
        )

        self.layer3 = nn.Sequential(
            BasicBlock(128, 256, 2, nn.Sequential(
                nn.Conv2d(128, 256, 1, 2, bias=False),
                nn.BatchNorm2d(256)
            )),
            BasicBlock(256, 256)
        )

        self.layer4 = nn.Sequential(
            BasicBlock(256, 512, 2, nn.Sequential(
                nn.Conv2d(256, 512, 1, 2, bias=False),
                nn.BatchNorm2d(512)
            )),
            BasicBlock(512, 512)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [7]:
# =========================
# 7. Training Setup
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ResNet18(num_classes).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 8. Training Loop
# =========================

def train_one_epoch():
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [8]:
# =========================
# 9. Evaluation
# =========================

def evaluate():
    model.eval()
    total_loss = 0

    total_correct = 0
    total_elements = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Convert logits → probabilities → binary predictions
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            total_correct += (preds == labels).sum().item()
            total_elements += labels.numel()

    val_loss = total_loss / len(test_loader)
    val_acc = total_correct / total_elements

    return val_loss, val_acc

In [12]:
# =========================
# 10. Run Training
# =========================
from sklearn.metrics import classification_report, multilabel_confusion_matrix, f1_score
import numpy as np
import matplotlib.pyplot as plt

def plot_conf_matrix(cm, genre):
    fig, ax = plt.subplots()
    ax.imshow(cm)

    ax.set_title(genre)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    plt.show()

def get_predictions():
    model.eval()

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.3).int()

            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)

    return all_labels, all_preds
    
EPOCHS = 10
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_one_epoch()
    val_loss, val_acc = evaluate()

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_resnet18_steam.pth")
        print("✅ Saved new best model")

print("Training complete.")
model.load_state_dict(torch.load("best_resnet18_steam.pth"))
model.to(device)
model.eval()
y_true, y_pred = get_predictions()
idx_to_genre = {i: g for g, i in genre_to_idx.items()}
target_names = [idx_to_genre[i] for i in range(len(idx_to_genre))]
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
mcm = multilabel_confusion_matrix(y_true, y_pred)
for i, genre in enumerate(target_names):
    tn, fp, fn, tp = mcm[i].ravel()
    print(f"\nGenre: {genre}")
    print(f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

Epoch 1/10 | Train Loss: 0.1684 | Val Loss: 0.1663 | Val Acc: 0.9260
✅ Saved new best model
Epoch 2/10 | Train Loss: 0.1657 | Val Loss: 0.1649 | Val Acc: 0.9262
✅ Saved new best model
Epoch 3/10 | Train Loss: 0.1647 | Val Loss: 0.1638 | Val Acc: 0.9267
✅ Saved new best model
Epoch 4/10 | Train Loss: 0.1641 | Val Loss: 0.1638 | Val Acc: 0.9265
✅ Saved new best model
Epoch 5/10 | Train Loss: 0.1634 | Val Loss: 0.1641 | Val Acc: 0.9263
Epoch 6/10 | Train Loss: 0.1629 | Val Loss: 0.1623 | Val Acc: 0.9274
✅ Saved new best model
Epoch 7/10 | Train Loss: 0.1624 | Val Loss: 0.1624 | Val Acc: 0.9272
Epoch 8/10 | Train Loss: 0.1620 | Val Loss: 0.1617 | Val Acc: 0.9280
✅ Saved new best model
Epoch 9/10 | Train Loss: 0.1615 | Val Loss: 0.1618 | Val Acc: 0.9271
Epoch 10/10 | Train Loss: 0.1610 | Val Loss: 0.1612 | Val Acc: 0.9281
✅ Saved new best model
Training complete.
                       precision    recall  f1-score   support

            360 Video       0.00      0.00      0.00         0
  

/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# Classification report and confusion matrix stats
model.load_state_dict(torch.load("best_resnet18_steam.pth"))
model.to(device)
model.eval()
y_true, y_pred = get_predictions()
idx_to_genre = {i: g for g, i in genre_to_idx.items()}
target_names = [idx_to_genre[i] for i in range(len(idx_to_genre))]
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
mcm = multilabel_confusion_matrix(y_true, y_pred)
for i, genre in enumerate(target_names):
    tn, fp, fn, tp = mcm[i].ravel()
    print(f"\nGenre: {genre}")
    print(f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))